## Search Engine With Tolls And Agents 

In [1]:
## Arxiv --Research
##tolls creation

from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

In [2]:
## Used the inbuilt tool of wikipedia
api_warpper_wiki = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_warpper_wiki)
wiki.name

'wikipedia'

In [3]:
api_warpper_arxiv = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_warpper_arxiv)
arxiv.name

'arxiv'

In [4]:
tools=[wiki,arxiv]

In [5]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS 
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
loader = WebBaseLoader("https://api.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectorstore= FAISS.from_documents(documents, OpenAIEmbeddings())
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C6FB7290D0>, search_kwargs={})

In [9]:
from langchain_core.tools import create_retriever_tool
retriever_Tool=create_retriever_tool(retriever,"langsmith-search","Search any information about langsmith")

retriever_Tool.name

'langsmith-search'

In [10]:
tools=[wiki,arxiv,retriever_Tool]

In [11]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Langchain\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.arxiv.ArxivError'>, <class 'arxiv.arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001C6FE87E2A0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001C6FE87C220>)]

In [12]:
## Run all this tools with agetns and llm models

## tools,llm-->AgentExecutor
import os 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import openai
load_dotenv()

# Load API keys
groq_api_key = os.getenv("GROQ_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")


# Check API keys
if not groq_api_key:
    st.error("GROQ_API_KEY is not set in .env")

if not openai_api_key:
    st.error("OPENAI_API_KEY is not set in .env")


# LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model="openai/gpt-oss-20b"
)

In [19]:
from langchainhub import Client

client = Client()

prompt = client.pull("hwchase17/openai-functions-agent")

prompt

C:\Users\Nitro-Silver\AppData\Local\Temp\ipykernel_19588\481222725.py:5: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("hwchase17/openai-functions-agent")
c:\Langchain\.venv\Lib\site-packages\langchainhub\client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


'{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "SystemMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are a helpful assistant", "input_variables": [], "template_format": "f-string", "partial_variables": {}}}}}, {"id": ["langchain", "prompts", "chat", "MessagesPlaceholder"], "lc": 1, "type": "constructor", "kwargs": {"optional": true, "variable_name": "chat_history"}}, {"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "{input}", "input_variables": ["input"], "template_format": "f-string", "partial_variables": {}}}}}, {"id": ["langchain", "pr

In [21]:
## Agents
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=[retriever_tool]
)

NameError: name 'retriever_tool' is not defined

In [22]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=tools
)

C:\Users\Nitro-Silver\AppData\Local\Temp\ipykernel_19588\703245479.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [24]:
response = agent.invoke(
    {
        "messages": [
            ("user", "Search for information about LangSmith")
        ]
    }
)

response

{'messages': [HumanMessage(content='Search for information about LangSmith', additional_kwargs={}, response_metadata={}, id='7a66d567-ff34-4164-bd9b-12b850755219'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use langsmith-search function.', 'tool_calls': [{'id': 'fc_d6c61680-be44-43ba-817a-8d3addc4f328', 'function': {'arguments': '{"query":"LangSmith"}', 'name': 'langsmith-search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 266, 'total_tokens': 301, 'completion_time': 0.050196304, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.014048655, 'prompt_tokens_details': None, 'queue_time': 0.472620694, 'total_time': 0.064244959}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_639a5351c8', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a042bb-ca54-73e0-bfb0-46e2af636022-0', tool_calls=[{'name': 

In [25]:
response["messages"][-1].content


'**LangSmith – The LLM‑Application Observability Platform**\n\n| Feature | What it does | Why it matters |\n|---------|--------------|----------------|\n| **Tracing** | Records every request/response that flows through an LLM‑powered pipeline (LLMs, tools, memory, prompts, callbacks, etc.). | Gives a complete “execution history” so developers can replay, debug, and audit runs. |\n| **Metrics & Dashboards** | Aggregates latency, token usage, cost, error rates, and custom metrics. | Enables performance monitoring and cost‑control for production workloads. |\n| **Debugging UI** | Interactive visual interface that shows prompt templates, tool calls, and chain execution. | Lets engineers spot bugs in prompt design, tool integration, or data flow. |\n| **Evaluation & Benchmarking** | Built‑in tools to run benchmark suites (e.g., MMLU, BIG-Bench) and compare models or prompt variants. | Provides objective, repeatable metrics for model selection and fine‑tuning. |\n| **Observability APIs** | S

In [26]:
response = agent.invoke(
    {
        "messages": [
            ("user", "what is the attention")
        ]
    }
)

response

{'messages': [HumanMessage(content='what is the attention', additional_kwargs={}, response_metadata={}, id='5b484a51-477e-4a6f-9521-d000e2f27ee1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what is the attention". Likely about "attention" in deep learning or maybe general. Provide explanation. Could use Wikipedia. Let\'s search.', 'tool_calls': [{'id': 'fc_1f1222b5-fb9c-459c-b604-7e21044e5263', 'function': {'arguments': '{"query":"attention mechanism deep learning"}', 'name': 'wikipedia'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 264, 'total_tokens': 322, 'completion_time': 0.06398615, 'completion_tokens_details': {'reasoning_tokens': 33}, 'prompt_time': 0.015010011, 'prompt_tokens_details': None, 'queue_time': 0.373969485, 'total_time': 0.078996161}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs

In [27]:
response["messages"][-1].content

'**Attention** is a technique in machine learning—especially in deep learning for sequence‑based tasks (e.g., NLP, computer vision, speech)—that lets a model *focus* on the most relevant parts of its input when making a prediction.\n\n### Core idea\nWhen processing a sequence (words, image patches, audio frames, etc.), the model computes a *weight* for each element that reflects how important that element is to the current output. These weights are called **attention scores**. The model then forms a weighted sum of the input representations, giving more influence to the important elements.\n\n### How it works (simplified)\n1. **Queries, keys, values**  \n   - Each input token (or element) is projected into three vectors:  \n     - **Query** (what we’re looking for)  \n     - **Key** (what each element offers)  \n     - **Value** (the actual content to be combined).  \n\n2. **Score calculation**  \n   - For a given query, compute a dot‑product (or other similarity measure) with each key